<a href="https://colab.research.google.com/github/cpelizza/CLAP/blob/main/CLAP_lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Libraries

In [ ]:
import wandb

In [ ]:
!pip install python-dotenv

In [ ]:
import os
os.environ["WANDB_API_KEY"] = "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"

In [ ]:
#import dotenv
#dotenv.load_dotenv("/content/env")

In [ ]:
!gsutil cp "gs://3d-shapes/3dshapes.h5" .

Copying gs://3d-shapes/3dshapes.h5...
==> NOTE: You are downloading one or more large file(s), which would
run significantly faster if you enabled sliced object downloads. This
feature is enabled by default but requires that compiled crcmod be
installed (see "gsutil help crcmod").

\ [1 files][255.2 MiB/255.2 MiB]                                                
Operation completed over 1 objects/255.2 MiB.                                    


In [ ]:
pip install lightning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 29.0 MB/s eta 0:00:00


In [ ]:
# =========================
# Standard library
# =========================
import random
import logging
import logging.handlers
import pickle as pkl
from argparse import ArgumentParser
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

# =========================
# Third-party libraries
# =========================
import h5py
import numpy as np
import pandas as pd
from six.moves import range

# =========================
# PyTorch
# =========================
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.distributions import Normal
from torch.utils.data import DataLoader, Dataset, Subset
from torch.utils.data.sampler import Sampler
from torch.utils.tensorboard import SummaryWriter

# =========================
# Torchvision
# =========================
from torchvision.transforms import Compose, Grayscale, Normalize, Resize, ToTensor

# =========================
# Scikit-learn
# =========================
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# =========================
# Lightning
# =========================
import lightning as L
from lightning.pytorch import LightningDataModule
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch import seed_everything

# =========================
# Dataclasses
# =========================
from dataclasses import dataclass, field, fields

# UTILS

In [ ]:

DATA_DIR = Path("./data").resolve()


class SubsetSampler(Sampler):
    def __init__(self, num_samples, n):
        self.num_samples = num_samples
        self.list_n = np.arange(n)

    def __iter__(self):
        random.shuffle(self.list_n)
        return iter(self.list_n[: self.num_samples].tolist())

    def __len__(self):
        return self.num_samples


# The following are utilities for datasets where the ground truth factors are known.
# https://github.com/google-research/disentanglement_lib/blob/86a644d4ed35c771560dc3360756363d35477357/disentanglement_lib/data/ground_truth/ground_truth_data.py#L22
class GroundTruthData(object):
    """Abstract class for data sets that are two-step generative models.
    It defines how datasets with known generative factors behave.
    """

    @property
    def num_factors(self):
        raise NotImplementedError()

    @property
    def factor_bases(self):
        raise NotImplementedError()

    def sample_factors(self, num, random_state=None):
        """Sample a batch of latent factors Y."""
        raise NotImplementedError()

    def sample_observations_from_factors(
        self, factors, random_state=None, draw_label=False
    ):
        """Sample a batch of observations X given a batch of factors Y."""
        raise NotImplementedError()

    def sample(self, num, random_state=None, draw_label=False):
        """Sample a batch of factors Y and observations X."""
        factors = self.sample_factors(num, random_state)
        return factors, self.sample_observations_from_factors(
            factors, random_state, draw_label=draw_label
        )

    def sample_observations(self, num, random_state=None, draw_label=False):
        """Sample a batch of observations X."""
        return self.sample(num, random_state, draw_label=draw_label)[1]

    def idx_to_factors(self, idx):
        """
        maps a linear index to its multi-dimensional factor configuration
        """
        factors = np.zeros(shape=(self.num_factors,))
        for pos, factor_base in enumerate(self.factor_bases):
            factor = np.floor_divide(idx, factor_base)
            factors[pos] = factor
            idx -= factor * factor_base
        assert idx == 0, str(idx) + " remainder is not 0"
        return factors


class SplitDiscreteStateSpace:
    def __init__(self, factor_sizes):
        self.factor_sizes = factor_sizes
        self.num_factors = len(factor_sizes)

    def sample_latent_factors(self, num):
        """Sample a batch of the latent factors."""
        factors = np.zeros(shape=(num, self.num_factors), dtype=np.int64)
        for i in range(self.num_factors):
            factors[:, i] = self._sample_factor(i, num)
        return factors

    def _sample_factor(self, i, num):
        return np.random.randint(
            low=0, high=self.factor_sizes[i], size=(num,)
        )  # high is exclusive


class StateSpaceAtomIndex(object):
    """Index mapping from features to positions of state space atoms."""

    def __init__(self, factor_sizes, features):
        """Creates the StateSpaceAtomIndex.
        Args:
          factor_sizes: List of integers with the number of distinct values for each
            of the factors.
          features: Numpy matrix where each row contains a different factor
            configuration. The matrix needs to cover the whole state space.
        """
        self.factor_sizes = factor_sizes
        num_total_atoms = np.prod(self.factor_sizes)
        self.factor_bases = num_total_atoms / np.cumprod(self.factor_sizes)
        feature_state_space_index = self._features_to_state_space_index(features)
        if np.unique(feature_state_space_index).size != num_total_atoms:
            raise ValueError("Features matrix does not cover the whole state space.")
        lookup_table = np.zeros(num_total_atoms, dtype=np.int64)
        lookup_table[feature_state_space_index] = np.arange(num_total_atoms)
        self.state_space_to_save_space_index = lookup_table

    def features_to_index(self, features):
        """Returns the indices in the input space for given factor configurations.
        Args:
          features: Numpy matrix where each row contains a different factor
            configuration for which the indices in the input space should be
            returned.
        """
        state_space_index = self._features_to_state_space_index(features)
        return self.state_space_to_save_space_index[state_space_index]

    def _features_to_state_space_index(self, features):
        """Returns the indices in the atom space for given factor configurations.
        Args:
          features: Numpy matrix where each row contains a different factor
            configuration for which the indices in the atom space should be
            returned.
        """
        if np.any(features > np.expand_dims(self.factor_sizes, 0)) or np.any(
            features < 0
        ):
            raise ValueError("Feature indices have to be within [0, factor_size-1]!")
        return np.array(np.dot(features, self.factor_bases), dtype=np.int64)

# Load dataset

In [ ]:
!pip install datasets

In [ ]:
import requests
import gzip
import os

print("Downloading small_norb.")

output_dir = "small_norb"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

def download_and_gunzip(url, output_path):
    print(f"Downloading {url} to {output_path}.gz")
    response = requests.get(url, stream=True)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)

    with open(output_path + ".gz", 'wb') as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Decompressing {output_path}.gz")
    with gzip.open(output_path + ".gz", 'rb') as f_in:
        with open(output_path, 'wb') as f_out:
            f_out.write(f_in.read())
    os.remove(output_path + ".gz") # Remove the compressed file after decompression

files_to_download = [
    ("smallnorb-5x46789x9x18x6x2x96x96-training-dat.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x46789x9x18x6x2x96x96-training-dat.mat.gz"),
    ("smallnorb-5x46789x9x18x6x2x96x96-training-cat.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x46789x9x18x6x2x96x96-training-cat.mat.gz"),
    ("smallnorb-5x46789x9x18x6x2x96x96-training-info.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x46789x9x18x6x2x96x96-training-info.mat.gz"),
    ("smallnorb-5x01235x9x18x6x2x96x96-testing-dat.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x01235x9x18x6x2x96x96-testing-dat.mat.gz"),
    ("smallnorb-5x01235x9x18x6x2x96x96-testing-cat.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x01235x9x18x6x2x96x96-testing-cat.mat.gz"),
    ("smallnorb-5x01235x9x18x6x2x96x96-testing-info.mat", "https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/smallnorb-5x01235x9x18x6x2x96x96-testing-info.mat.gz")
]

for filename, url in files_to_download:
    file_path = os.path.join(output_dir, filename)
    if not os.path.exists(file_path):
        download_and_gunzip(url, file_path)
    else:
        print(f"File already exists: {file_path}")

print("Downloading small_norb completed!")

Decompressing small_norb/smallnorb-5x46789x9x18x6x2x96x96-training-dat.mat.gz
Decompressing small_norb/smallnorb-5x46789x9x18x6x2x96x96-training-cat.mat.gz
Decompressing small_norb/smallnorb-5x46789x9x18x6x2x96x96-training-info.mat.gz
Decompressing small_norb/smallnorb-5x01235x9x18x6x2x96x96-testing-dat.mat.gz
Decompressing small_norb/smallnorb-5x01235x9x18x6x2x96x96-testing-cat.mat.gz
Decompressing small_norb/smallnorb-5x01235x9x18x6x2x96x96-testing-info.mat.gz


In [ ]:
# from
# https://github.com/google-research/disentanglement_lib/blob/86a644d4ed35c771560dc3360756363d35477357/disentanglement_lib/data/ground_truth/norb.py

"""SmallNORB dataset."""
from __future__ import absolute_import, division, print_function

import os

import PIL
import numpy as np
import tensorflow.compat.v1 as tf
import torch
from six.moves import range
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset



SMALLNORB_TEMPLATE = os.path.join("/content","small_norb", "smallnorb-{}-{}.mat")
SMALLNORB_CHUNKS = ["5x46789x9x18x6x2x96x96-training", "5x01235x9x18x6x2x96x96-testing"]


class SmallNORB(GroundTruthData):
    """SmallNORB dataset.
    The data set can be downloaded from
    https://cs.nyu.edu/~ylclab/data/norb-v1.0-small/. Images are resized to 64x64.
    The ground-truth factors of variation are:
    0 - category (5 different values)
    1 - elevation (9 different values)
    2 - azimuth (18 different values)
    3 - lighting condition (6 different values)
    The instance in each category is randomly sampled when generating the images.
    """

    def __init__(self, transforms):
        self.images, features = _load_small_norb_chunks(
            SMALLNORB_TEMPLATE, SMALLNORB_CHUNKS
        )
        self.transforms = transforms
        self.factor_sizes = [5, 10, 9, 18, 6]
        # Instances are not part of the latent space.
        self.latent_factor_indices = [0, 1, 2, 3, 4]
        self.num_total_factors = features.shape[1]
        self.index = StateSpaceAtomIndex(self.factor_sizes, features)
        self.state_space = SplitDiscreteStateSpace(self.factor_sizes)

    @property
    def factors_num_values(self):
        return [self.factor_sizes[i] for i in self.latent_factor_indices]

    @property
    def observation_shape(self):
        return [64, 64, 1]

    def sample_factors(self, num, random_state=None):
        """Sample a batch of factors Y."""
        return self.state_space.sample_latent_factors(num)

    def sample_observations_from_factors(
        self, factors, random_state=None, draw_label=False
    ):
        # all_factors = self.state_space.sample_all_factors(factors, random_state)
        inds = self.index.features_to_index(factors)
        n = len(factors)
        x = torch.empty(n, 1, 64, 64)
        for i in range(n):
            x[i] = self.transforms(
                np.reshape(self.images[inds[i]].astype(np.float32), (64, 64, 1))
            )
        out_dict = {"x": x}
        if draw_label:
            y = torch.empty(n, 4)
            for i in range(n):
                y[i] = torch.from_numpy(self.label_from_factors(factors[i]))
            out_dict["y"] = y
        return out_dict

    @staticmethod
    def label_from_factors(factors):
        object_type, background = (
            factors[1],
            factors[4],
        )  # 0 to 9 included, 0 to 5 included

        # factors should be flat
        y = np.empty(shape=(4,), dtype=np.int_)
        # label 1: vehicle + light background
        y[0] = 1 if object_type >= 5 and background >= 3 else 0
        # label 2: vehicle  + dark background
        y[1] = 1 if object_type >= 5 and background < 3 else 0
        # label 3: human/animal + light background
        y[2] = 1 if object_type < 5 or background >= 3 else 0
        # label 4: human/animal+dark background
        y[3] = 1 if background < 3 else 0
        return y


def _load_small_norb_chunks(path_template, chunk_names):
    """Loads several chunks of the small norb data set for final use."""
    list_of_images, list_of_features = _load_chunks(path_template, chunk_names)
    features = np.concatenate(list_of_features, axis=0)
    features[:, 3] = features[:, 3] / 2  # azimuth values are 0, 2, 4, ..., 24
    return np.concatenate(list_of_images, axis=0), features


def _load_chunks(path_template, chunk_names):
    """Loads several chunks of the small norb data set into lists."""
    list_of_images = []
    list_of_features = []
    for chunk_name in chunk_names:
        norb = _read_binary_matrix(path_template.format(chunk_name, "dat"))
        list_of_images.append(_resize_images(norb[:, 0]))
        norb_class = _read_binary_matrix(path_template.format(chunk_name, "cat"))
        norb_info = _read_binary_matrix(path_template.format(chunk_name, "info"))
        list_of_features.append(np.column_stack((norb_class, norb_info)))
    return list_of_images, list_of_features


def _read_binary_matrix(filename):
    """Reads and returns binary formatted matrix stored in filename."""
    with tf.gfile.GFile(filename, "rb") as f:
        s = f.read()
        magic = int(np.frombuffer(s, "int32", 1)[0]) # Explicitly get scalar
        ndim = int(np.frombuffer(s, "int32", 1, 4)[0]) # Explicitly get scalar
        eff_dim = max(3, ndim)
        raw_dims = np.frombuffer(s, "int32", eff_dim, 8)
        dims = []
        for i in range(0, ndim):
            dims.append(raw_dims[i])

        dtype_map = {
            507333717: "int8",
            507333716: "int32",
            507333713: "float",
            507333715: "double",
        }
        data = np.frombuffer(s, dtype_map[magic], offset=8 + eff_dim * 4)
    data = data.reshape(tuple(dims))
    return data


def _resize_images(integer_images):
    resized_images = np.zeros((integer_images.shape[0], 64, 64))
    for i in range(integer_images.shape[0]):
        image = PIL.Image.fromarray(integer_images[i, :, :])
        image = image.resize((64, 64), PIL.Image.LANCZOS)
        resized_images[i, :, :] = image
    return resized_images / 255.0


class SmallNORBDataset(Dataset):
    def __init__(self, groundtruth, train, test_size=0.1, random_state=12345):
        self.train = train
        self.groundtruth = groundtruth
        self.transforms = groundtruth.transforms
        if train:
            self.ind, _ = train_test_split(
                range(len(groundtruth.images)),
                test_size=test_size,
                random_state=random_state,
            )
        else:
            _, self.ind = train_test_split(
                range(len(groundtruth.images)),
                test_size=test_size,
                random_state=random_state,
            )
        self.n = len(self.ind)
        # check whether index is in allowed set ( train / test set)
        self.check_ind = torch.zeros(len(self.groundtruth.images))
        self.check_ind[self.ind] = 1

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        while True:
            factors = self.groundtruth.sample_factors(1)
            ind = self.groundtruth.index.features_to_index(factors)
            factors = factors[0]
            # check if image in train / test set
            if self.check_ind[ind] == 1:
                x = self.groundtruth.images[ind].astype(np.float32)
                x = self.transforms(np.reshape(x, (64, 64, 1)))
                label = self.groundtruth.label_from_factors(factors)
                return x, label

    @property
    def factor_data(self):
        return self.groundtruth

In [ ]:
#from .chestxray import ChestXRay
#from .mpi import MPIData, MPIDataset
#from .plant_village import PlantVillageDataset
#from .shapes3d import Shapes3D, Shapes3DDataset
#from .small_norb import SmallNORB, SmallNORBDataset

dataset_statistics = {
    # dataset   # mean           # std
    "MPI": [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]],
    "SmallNORB": [[0.0], [1.0]],
    "Shapes3D": [[0.0, 0.0, 0.0], [1.0, 1.0, 1.0]],
    "PlantVillage": [
        [0.0, 0.0, 0.0],
        [1.0, 1.0, 1.0],
    ],
    "ChestXRay": [[0.0], [1.0]],
}


# Remark: while we use the following transformations at loading time, preprocessing some datasets yields a big speedup
# in terms of training time.
def vanilla_transform(dataset_name: str) -> Compose:
    return Compose(
        [
            ToTensor(),
            Normalize(*dataset_statistics[dataset_name]),
        ]
    )


def resize_transform(dataset_name: str, in_dim: int, n_channels=3) -> Compose:
    size = (in_dim, in_dim) if n_channels == 3 else in_dim
    return Compose(
        [
            ToTensor(),
            Resize(size=size),
            Normalize(*dataset_statistics[dataset_name]),
        ]
    )


def get_datasets(dataset_name: str) -> Tuple[Dataset, Dataset, int, int, int]:
    if dataset_name == "MPI":
        transforms = vanilla_transform(dataset_name)
        factor_data = MPIData(
            transforms=transforms,
            save_labels_and_factors=False,
        )
        train_dataset = MPIDataset(factor_data, train=True)
        test_dataset = MPIDataset(factor_data, train=False)
        n_channels, image_dim, n_classes = 3, 64, 4
    elif dataset_name == "SmallNORB":
        transforms = vanilla_transform(dataset_name)
        factor_data = SmallNORB(transforms)
        train_dataset = SmallNORBDataset(factor_data, train=True)
        test_dataset = SmallNORBDataset(factor_data, train=False)
        n_channels, image_dim, n_classes = 1, 64, 4
    elif dataset_name == "Shapes3D":
        transforms = resize_transform(dataset_name, 64, 3)
        factor_data = Shapes3D(transforms)
        train_dataset = Shapes3DDataset(
            train=True, ground_truth_data=factor_data, test_size=0.4
        )
        test_dataset = Shapes3DDataset(
            train=False, ground_truth_data=factor_data, test_size=0.4
        )
        n_channels, image_dim, n_classes = 3, 64, 4
    elif dataset_name == "PlantVillage":
        transforms = resize_transform(dataset_name, 64)
        dataset = PlantVillageDataset(
            "Tomato",
            transforms=transforms,
        )
        train_idx, test_idx = train_test_split(
            np.arange(len(dataset)), train_size=0.9, random_state=1234
        )
        train_dataset = Subset(dataset, train_idx)
        test_dataset = Subset(dataset, test_idx)
        n_channels, image_dim, n_classes = 3, 64, 10
    elif dataset_name == "ChestXRay":
        transforms = Compose(
            [
                # some images have 1 channel, others 4. Make them all 1 channel.
                Grayscale(num_output_channels=1),
                resize_transform(dataset_name, 64),
            ]
        )
        train_dataset = ChestXRay(
            train=True,
            transforms=transforms,
        )
        test_dataset = ChestXRay(
            train=False,
            transforms=transforms,
        )
        n_channels, image_dim, n_classes = 3, 64, 15
    else:
        raise NotImplementedError(f"Dataset {dataset_name} unknown.")

    # store mean and variance as dataset attributes. Ugly but needed.
    # Same with factor indices.
    for dataset in (train_dataset, test_dataset):
        setattr(dataset, "mean", dataset_statistics[dataset_name][0])
        setattr(dataset, "std", dataset_statistics[dataset_name][1])

    return train_dataset, test_dataset, n_channels, image_dim, n_classes

# CFG

In [ ]:

@dataclass
class DatasetConfig:
  n_channels: int
  image_dim: int
  n_classes: int
  dataset_name: str

@dataclass
class Shapes3D_config(DatasetConfig):
  n_channels: int = 3
  image_dim: int = 64
  n_classes: int = 4
  dataset_name: str = "Shapes3D"

@dataclass
class SmallNORB_config(DatasetConfig):
  n_channels: int = 1
  image_dim: int = 64
  n_classes: int = 4
  dataset_name: str = "SmallNORB"

@dataclass
class Config:
    seed: int = field(default = None, metadata={"help":"Random seed."})
    #dataset: str =  field(default = "MPI", metadata = {help:"Dataset used for the experiment. Available: MPI, Shapes3D, SmallNORB, PlantVillage, ChestXRay"})
    dataset: DatasetConfig = field(default_factory = SmallNORB_config, metadata = {help:"Dataset used for the experiment. Available: MPI, Shapes3"})

    z_core_dim: int = field(default = 10, metadata = {help: "Dimension of the core latent space in the model."})
    z_style_dim: int = field(default = 20, metadata = {help:"Dimension of the style latent space in the model."})
    y_reg: float = field(default = 50, metadata = {help:"Weight of the prediction error during training."})
    group_sparsity_reg: float = field(default = 0.5, metadata = {help: "Weight of the sparsity regularization term during training."})

    lr: float = field(default = 1.0e-4, metadata = {help:"Optimizer learning rate."})
    batch_size: int = field(default = 123, metadata = {help:"Batch size."})

    n_epochs_start: int = field(default = 200, metadata = {help:"Initial number of epochs during training. Used for scheduling of the regularization parameters."})
    n_epochs_beta: int = field(default = 400, metadata = {help:"Middle number of epochs during training. Used for scheduling of the regularization parameters."})
    n_epochs_end: int = field(default = 200, metadata = {help: "Final number of epochs during training. Used for scheduling of the regularization parameters."})

    save_path: Path = field(default = None, metadata = {help: "Path of the directory where training results are saved."})

    wandb_project: str = field(default = "CLAP", metadata = {help: "Name of the wandb project."})
    wandb_run_name: str = field(default = "test", metadata = {help: "Name of the wandb run."})


# NN definition

In [ ]:
"""Module for the neural network architectures utilized in CLAP. """
class CLAPEncoderBackbone(nn.Module):
      """
      The shared encoder backbone, used for both Predction VAE and Concept Learning VAE.
      """
      def __init__(
          self, n_channels: int, in_dim: int, intermediate_size: int = 256
      ) -> None:
          super().__init__()
          self.n_channels = n_channels
          self.in_dim = in_dim
          self.intermediate_size = intermediate_size

          # https://arxiv.org/pdf/1912.00155.pdf
          self.fc = nn.Sequential(
              nn.Conv2d(self.n_channels, 64, kernel_size=3, stride=2, padding=1),
              nn.LeakyReLU(inplace=True),
              nn.Dropout2d(p=0.1),
              nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
              nn.LeakyReLU(inplace=True),
              nn.Dropout2d(p=0.1),
              nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
              nn.LeakyReLU(inplace=True),
              nn.Dropout2d(p=0.1),
              nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1),
              nn.LeakyReLU(inplace=True),
              nn.Dropout2d(p=0.1),
              nn.Flatten(),
              nn.Linear(1024, self.intermediate_size),
              nn.ReLU(inplace=True),
          )

      def forward(self, x: torch.Tensor) -> torch.Tensor:
          encoded = self.fc(x)
          return encoded


class CLAPDecoder(nn.Module):
    def __init__(self, n_channels: int, z_core_dim: int, z_style_dim: int) -> None:
        super().__init__()
        self.z_dim = z_core_dim + z_style_dim
        self.n_channels = n_channels

        # https://arxiv.org/pdf/1912.00155.pdf
        self.fc = nn.Sequential(
            nn.Linear(self.z_dim, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, 1024),
            View(-1, 64, 4, 4),
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=0),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, self.n_channels, kernel_size=4, stride=2, padding=2),
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        decoded = self.fc(z)
        return decoded

    def get_first_linear_layer(self) -> torch.Tensor:
        return next(self.fc.parameters())


class LinearClassifier(nn.Module):
    def __init__(self, in_dim: int, n_classes: int) -> None:
        super().__init__()
        self.fc = nn.Linear(in_dim, n_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    def get_weights(self) -> torch.Tensor:
        return next(self.fc.parameters())


class View(nn.Module):
    def __init__(self, *dim):
        super().__init__()
        self.dim = dim

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x.view(self.dim)

# Main model

In [ ]:
class CLAP(nn.Module):
    """CLAP architecture.
    CLAP is composed of two partially overlapping VAEs: a prediction VAE and a concept learning VAE. These two VAEs
    share the same decoder. The encoder is shared partially:
    - a shared backbone structure maps images x to the style latent features of both VAEs
    - another encoder maps x to the core latent features of the prediction VAE
    - another encoder maps x and y to the core latent features of the concept learning VAE
    All these encoders have a very similar backbone structure, with linear layers added to adjust for the different
    input or output dimensions.
    """

    def __init__(
        self,
        n_channels: int,
        in_dim: int,
        z_style_dim: int,
        z_core_dim: int,
        n_outputs: int,
        intermediate_size=256,
    ) -> None:
        """
        :param n_channels: number of channels in the input images
        :param in_dim: dimension of the input images, which have then shape (batch_size, n_channels, in_dim, in_dim)
        :param z_style_dim: dimension of the style latent space
        :param z_core_dim: dimension of the core latent space
        :param n_outputs: number of binary labels in the supervision y, which has shape (batch_size, n_outputs)
        :param intermediate_size: intermediate size, output of the encoder backbone shared by the prediction and
        concept learning VAEs
        """
        super().__init__()

        self.n_channels = n_channels
        self.in_dim = in_dim
        self.z_style_dim = z_style_dim
        self.z_core_dim = z_core_dim
        self.n_outputs = n_outputs
        self.intermediate_size = intermediate_size

        (
            self.decoder,  # shared decoder
            self.cl_vae,  # concept learning VAE
            self.pred_vae,  # prediction VAE
        ) = self.construct_clap_parts()

    def construct_clap_parts(self) -> Tuple[nn.Module, nn.Module, nn.Module]:
        shared_encoder_backbone = CLAPEncoderBackbone(self.n_channels, self.in_dim)
        shared_style_mean_layer = nn.Linear(self.intermediate_size, self.z_style_dim)
        shared_style_log_var_layer = nn.Linear(self.intermediate_size, self.z_style_dim)

        cl_core_encoder_backbone = nn.Sequential(
            CLAPEncoderBackbone(self.n_channels, self.in_dim, self.intermediate_size),
            nn.Linear(self.intermediate_size, 32),
            nn.ReLU(nn.ReLU(inplace=True)),
            nn.Dropout(p=0.05),
        )
        cl_core_mean_layer = nn.Linear(32 + self.n_outputs, self.z_core_dim)
        cl_core_log_var_layer = nn.Linear(32 + self.n_outputs, self.z_core_dim)

        decoder = CLAPDecoder(self.n_channels, self.z_core_dim, self.z_style_dim)

        # prediction (P) part of CLAP
        pred_vae = PredictionVAE(
            encoder_backbone=shared_encoder_backbone,
            style_mean_layer=shared_style_mean_layer,
            style_log_var_layer=shared_style_log_var_layer,
            core_mean_layer=nn.Linear(self.intermediate_size, self.z_core_dim),
            core_log_var_layer=nn.Linear(self.intermediate_size, self.z_core_dim),
            decoder=decoder,
            predictor=LinearClassifier(self.z_core_dim, self.n_outputs),
        )

        # concept learning (CL) part of CLAP
        cl_vae = ConceptLearningVAE(
            style_encoder_backbone=shared_encoder_backbone,
            core_encoder_backbone=cl_core_encoder_backbone,
            style_mean_layer=shared_style_mean_layer,
            style_log_var_layer=shared_style_log_var_layer,
            core_mean_layer=cl_core_mean_layer,
            core_log_var_layer=cl_core_log_var_layer,
            decoder=decoder,
            n_outputs=self.n_outputs,
        )

        return decoder, cl_vae, pred_vae

    def get_decoder_first_linear_layer(self) -> torch.Tensor:
        return self.decoder.get_first_linear_layer()

    def get_prediction_weights(self) -> torch.Tensor:
        return self.pred_vae.predictor.get_weights()

    def forward(
        self, x: torch.Tensor, y: torch.Tensor
    ) -> Dict[str, Dict[str, torch.Tensor]]:
        pred_out = self.pred_vae(x)
        cl_out = self.cl_vae(x, y)

        return {
            "pred": pred_out,
            "cl": cl_out,
        }


class ConceptLearningVAE(nn.Module):
    def __init__(
        self,
        style_encoder_backbone: nn.Module,
        core_encoder_backbone: nn.Module,
        style_mean_layer: nn.Module,
        style_log_var_layer: nn.Module,
        core_mean_layer: nn.Module,
        core_log_var_layer: nn.Module,
        decoder: nn.Module,
        n_outputs: int,
    ) -> None:
        super().__init__()
        self.n_outputs = n_outputs

        self.style_encoder_backbone = style_encoder_backbone
        self.core_encoder_backbone = core_encoder_backbone

        # the following four layers maps the intermediate encoding of
        # self.style_encoder_backbone and self.core_encoder_backbone to
        # the mean and log-variance of the core and style latent features
        self.style_mean_layer = style_mean_layer
        self.style_log_var_layer = style_log_var_layer
        self.core_mean_layer = core_mean_layer
        self.core_log_var_layer = core_log_var_layer

        self.decoder = decoder

        (
            self.core_prior_mean,
            self.core_prior_log_var,
        ) = self._register_learnable_priors()

    def _register_learnable_priors(self) -> Tuple[nn.Module, nn.Module]:
        """Create the neural network layers that represent the conditional prior p(z | y) for the core features."""
        z_core_dim = next(self.core_mean_layer.parameters()).shape[0]

        mean_cl = nn.Linear(self.n_outputs, z_core_dim)
        log_var_cl = nn.Linear(self.n_outputs, z_core_dim)
        return mean_cl, log_var_cl

    def forward(self, x: torch.Tensor, y: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ENCODING
        # style features
        encoded_x_style = self.style_encoder_backbone(x)
        mean_style = self.style_mean_layer(encoded_x_style)
        log_var_style = self.style_log_var_layer(encoded_x_style)
        z_style = Normal(loc=mean_style, scale=torch.exp(0.5 * log_var_style)).rsample()

        # core features (learn priors for z_core basing on y)
        encoded_x_core = self.core_encoder_backbone(x)
        if len(y.shape) == 1:
            encoded_xy = torch.cat([encoded_x_core, y.unsqueeze(dim=-1)], dim=-1) #concatenate the label y
        else:
            encoded_xy = torch.cat([encoded_x_core, y], dim=-1)
        mean_core = self.core_mean_layer(encoded_xy)
        log_var_core = self.core_log_var_layer(encoded_xy)
        z_core = Normal(loc=mean_core, scale=torch.exp(0.5 * log_var_core)).rsample()

        # PRIOR computation
        float_y = y.type(torch.float)
        prior_mean_core = self.core_prior_mean(float_y)
        prior_log_var_core = self.core_prior_log_var(float_y)

        # RECONSTRUCTION of the input image
        z = torch.cat([z_core, z_style], dim=-1)
        reconstruction = self.decoder(z)

        return {
            "mean_core": mean_core,
            "log_var_core": log_var_core,
            "z_core": z_core,
            "mean_style": mean_style,
            "log_var_style": log_var_style,
            "z_style": z_style,
            "x_reconstructed": reconstruction,
            "prior_mean_core": prior_mean_core,
            "prior_log_var_core": prior_log_var_core,
        }


class PredictionVAE(nn.Module):
    def __init__(
        self,
        encoder_backbone: nn.Module,
        style_mean_layer: nn.Module,
        style_log_var_layer: nn.Module,
        core_mean_layer: nn.Module,
        core_log_var_layer: nn.Module,
        decoder: nn.Module,
        predictor: nn.Module,
    ) -> None:
        super().__init__()
        self.encoder_backbone = encoder_backbone

        # the following four layers maps the intermediate encoding of
        # self.encoder_backbone to the mean and log-variance of the
        # core and style latent spaces
        self.style_mean_layer = style_mean_layer
        self.style_log_var_layer = style_log_var_layer
        self.core_mean_layer = core_mean_layer
        self.core_log_var_layer = core_log_var_layer

        self.decoder = decoder
        self.predictor = predictor

    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:
        # ENCODING the input into mean and style features
        encoded = self.encoder_backbone(x)

        mean_style = self.style_mean_layer(encoded)
        log_var_style = self.style_log_var_layer(encoded)
        z_style = Normal(loc=mean_style, scale=torch.exp(0.5 * log_var_style)).rsample()

        mean_core = self.core_mean_layer(encoded)
        log_var_core = self.core_log_var_layer(encoded)
        z_core = Normal(loc=mean_core, scale=torch.exp(0.5 * log_var_core)).rsample()

        # PREDICTION from core feature
        # Change the self.training attribute calling .eval() or .train() methods
        if self.training:
            y_pred = self.predictor(z_core)
        else:
            y_pred = self.predictor(mean_core)

        # RECONSTRUCTION of the input image
        z = torch.cat([z_core, z_style], dim=-1)
        reconstruction = self.decoder(z)

        return {
            "mean_core": mean_core,
            "log_var_core": log_var_core,
            "z_core": z_core,
            "mean_style": mean_style,
            "log_var_style": log_var_style,
            "z_style": z_style,
            "x_reconstructed": reconstruction,
            "y_pred": y_pred,
        }


def logits_to_labels(logit: torch.Tensor) -> torch.Tensor:
    """Convert tensor of logit values to binary labels."""
    return torch.where(logit > 0, 1, 0)

# Loss function

In [ ]:

def accuracy(
    y_pred: torch.Tensor, y: torch.Tensor
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Given a tensor y_pred of logits, predict the accuracy with respect to the ground truth labels y
    :param y_pred: tensor of shape (batch_size, n_labels)
    :param y: tensor of shape (batch_size, n_labels)
    :return: two tensors, one of shape (n_labels,) with the accuracy per label and one with
    the mean accuracy averaged over the batch and all the labels
    """
    # y_pred are predicted probabilities in case of binary class
    tmp = ((y_pred >= 0.5).long() == y).float()
    return torch.mean(tmp, dim=0), torch.mean(tmp)


def bernoulli_reconstruction_loss(
    reconstructed: torch.Tensor, x: torch.Tensor
) -> torch.Tensor:
    """Reconstruction loss for images rescaled in [0, 1]. The reconstruction contains the logits of the
    pixels, with shape (batch_size, *image_dimension), while x are the original images  in [0, 1]."""
    # input reconstructions are unnormalized logits
    return (
        F.binary_cross_entropy_with_logits(reconstructed, x, reduction="sum")
        / x.size()[0]
    )


def latent_kl_divergence(
    name_z_loss: str,
    model_out_dict: Dict[str, torch.Tensor],
    z_core_dim: int,
    z_style_dim: int,
) -> torch.Tensor:
    """Various KL-divergence implementations for core and style spaces of CLAP."""
    # standard kl prior for zcore and zstyle in prediction VAE
    if name_z_loss == "prior_z_pred":
        kl_div = iso_kl_div(
            torch.cat(
                [
                    model_out_dict["pred"]["mean_core"],
                    model_out_dict["pred"]["mean_style"],
                ],
                dim=-1,
            ),
            torch.cat(
                [
                    model_out_dict["pred"]["log_var_core"],
                    model_out_dict["pred"]["log_var_style"],
                ],
                dim=-1,
            ),
        )
    # standard kl prior for zcore in prediction VAE
    elif name_z_loss == "prior_z_core_pred":
        kl_div = iso_kl_div(
            model_out_dict["pred"]["mean_core"], model_out_dict["pred"]["log_var_core"]
        )
        kl_div *= z_core_dim / (z_core_dim + z_style_dim)
    # standard kl prior for zstyle in prediction VAE
    elif name_z_loss == "prior_z_style_pred":
        kl_div = iso_kl_div(
            model_out_dict["pred"]["mean_style"],
            model_out_dict["pred"]["log_var_style"],
        )
        kl_div *= z_style_dim / (z_core_dim + z_style_dim)
    # standard kl prior for zcore and zstyle in concept learning VAE
    elif name_z_loss == "prior_z_cl":
        kl_div = iso_kl_div(
            torch.cat(
                [
                    model_out_dict["cl"]["mean_core"],
                    model_out_dict["cl"]["mean_style"],
                ],
                dim=-1,
            ),
            torch.cat(
                [
                    model_out_dict["cl"]["log_var_core"],
                    model_out_dict["cl"]["log_var_style"],
                ],
                dim=-1,
            ),
        )
    # standard kl prior for zcore in concept learning VAE
    elif name_z_loss == "prior_z_core_cl":
        kl_div = iso_kl_div(
            model_out_dict["cl"]["mean_core"], model_out_dict["cl"]["log_var_core"]
        )
        kl_div *= z_core_dim / (z_core_dim + z_style_dim)
    # standard kl prior for zstyle in concept learning VAE
    elif name_z_loss == "prior_z_style_cl":
        kl_div = iso_kl_div(
            model_out_dict["cl"]["mean_style"], model_out_dict["cl"]["log_var_style"]
        )
        kl_div *= z_core_dim / (z_core_dim + z_style_dim)
    # kl div to learned prior dependent on y in concept learning VAE
    elif name_z_loss == "prior_z_core_y_cl":
        kl_div = learned_kl_div(
            model_out_dict["cl"]["mean_core"],
            model_out_dict["cl"]["log_var_core"],
            model_out_dict["cl"]["prior_mean_core"],
            model_out_dict["cl"]["prior_log_var_core"],
        )
        kl_div *= z_core_dim / (z_core_dim + z_style_dim)
    else:
        raise NotImplementedError("unknown kl divergence.")

    return kl_div / 2


def iso_kl_div(mean: torch.Tensor, log_var: torch.Tensor) -> torch.Tensor:
    """KL-divergence between a Gaussian, specified by its mean and log-variance, and a standard Gaussian."""
    loss = 0.5 * (mean.pow(2) + log_var.exp() - log_var - 1)
    return loss.sum(1).mean()


# kl div formula from here: https://eehsan.github.io/Notes/vae.pdf
def learned_kl_div(
    mean: torch.Tensor,
    log_var: torch.Tensor,
    prior_mean: torch.Tensor,
    prior_log_var: torch.Tensor,
) -> torch.Tensor:
    """KL-divergence between a posterior and a prior.
    Both are Gaussian distributions, with specified mean and log-variance.
    """
    n = mean.size()[1]
    m = mean.size()[0]
    mean_cond = prior_mean.expand(m, n)
    log_var_cond = prior_log_var.expand(m, n)
    loss = 0.5 * (
        ((mean - mean_cond).pow(2) + log_var.exp()) / log_var_cond.exp()
        - log_var
        + log_var_cond
        - 1
    )
    return loss.sum(1).mean()

# other functions

In [ ]:
def cycle_params(
    beta: float,
    y_reg: float,
    n_start: int,
    n_epochs_beta: int,
    n_end: int,
    n_cycle_beta: int,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Create the cyclical annealing schedules for:
    - the beta parameter (KL divergence multiplier in the ELBO)
    - the y_reg parameter (prediction loss multiplier in the ELBO)
    """
    beta_reg_values = torch.ones(n_start + n_epochs_beta + n_end)
    beta_reg_values[0:n_start] = 0
    beta_reg_values[n_start : n_epochs_beta + n_start] = frange_cycle_linear(
        0, 1, n_epochs_beta, n_cycle_beta
    )
    beta_reg_values *= beta

    y_reg_values = torch.ones(n_start + n_epochs_beta + n_end)
    y_reg_values[0 : n_start + n_epochs_beta] = frange_cycle_linear(
        0, 1, n_epochs_beta + n_start, 1
    )
    y_reg_values *= y_reg
    return beta_reg_values, y_reg_values

In [ ]:
def frange_cycle_linear(
    start: int, stop: int, n_epoch: int, n_cycle: int = 4, ratio: float = 0.8
) -> torch.Tensor:
    """Create the schedules for cyclical annealing."""
    L = torch.ones(n_epoch)
    period = n_epoch / n_cycle
    step = (stop - start) / (period * ratio)  # linear schedule

    for c in range(n_cycle):

        v, i = start, 0
        while v <= stop and (int(i + c * period) < n_epoch):
            L[int(i + c * period)] = v
            v += step
            i += 1
    return L


In [ ]:
list_base_loss = [
    "total_loss",
    "reconstruction_loss_pred",
    "reconstruction_loss_cl",
    "prediction_loss",
]
list_z_loss = [
    "prior_z_core_y_cl",
    "prior_z_style_cl",
    "prior_z_core_pred",
    "prior_z_style_pred",
]

class MetricTracker:
    def __init__(
        self,
        keys: List[Any],
        train_val_flag: bool,
        writer: Optional[SummaryWriter] = None,
    ) -> None:
        self.writer = writer
        self.train_val_flag = train_val_flag
        if "zcore_cl_pred" in keys:
            keys.append("zcore_cl_pred_cl")
            keys.append("zcore_cl_pred_pred")
        self._data = pd.DataFrame(index=keys, columns=["total", "counts", "average"])
        self._data_final = {key: [] for key in keys}
        self.reset()

    def reset(self) -> None:
        for col in self._data.columns:
            self._data[col].values[:] = 0

    def update(self, key: Any, value: Any, n: int = 1) -> None:
        self._data.total[key] += value * n
        self._data.counts[key] += n
        self._data.average[key] = self._data.total[key] / self._data.counts[key]

    def update_avg(self, n: int) -> None:
        for key in self._data_final:
            if self.writer is not None:
                value = self._data.average[key]
                self._data_final[key].append(value)
                self.writer.add_scalar(self.train_val_flag + key, value, n)

    def avg(self, key: Any) -> None:
        return self._data.average[key]

    def result(self) -> Dict[Any, Any]:
        return dict(self._data.average)

# Training

In [ ]:
class LightningCLAP(L.LightningModule):
    def __init__(self, cfg, train_dataset, test_dataset):
        super().__init__()
        self.cfg = cfg
        self.model = CLAP(cfg.dataset.n_channels, cfg.dataset.image_dim, self.cfg.z_style_dim, self.cfg.z_core_dim, cfg.dataset.n_classes)

        self.train_dataset = train_dataset
        self.test_dataset = test_dataset

        self.dev_mean = torch.tensor(self.train_dataset.mean, device="cuda")[
            None, :, None, None
        ]
        self.dev_std = torch.tensor(self.train_dataset.std, device="cuda")[
            None, :, None, None
        ]
        assert self.dev_mean.shape == self.dev_std.shape
        assert (
            self.dev_mean.shape[0]
            == self.dev_mean.shape[2]
            == self.dev_mean.shape[3]
            == 1
        )

        # loss functions (here I just save them, these functions will be used later)
        self.reconstruction_loss = bernoulli_reconstruction_loss
        self.loss_y = torch.nn.BCEWithLogitsLoss()
        self.accuracy = accuracy


        # attributes utilized during training
        self.group_sparsity_reg = cfg.group_sparsity_reg
        self.best_accuracy: float = 0.0
        self.beta_reg_values: Optional[torch.Tensor] = None
        self.y_reg_values: Optional[torch.Tensor] = None

        # schedules
        self.beta_reg_values, self.y_reg_values = cycle_params(
            1.0,
            cfg.y_reg,
            cfg.n_epochs_start,
            cfg.n_epochs_beta,
            cfg.n_epochs_end,
            2,
        )

        # Log model info
        logger.info(f"CLAPLightning initialized with:")
        logger.info(f"  z_core_dim: {cfg.z_core_dim}")
        logger.info(f"  z_style_dim: {cfg.z_style_dim}")
        logger.info(f"  n_classes: {cfg.dataset.n_classes}")

    def forward(self, inputs, target):
        return self.model(inputs, target)

    def training_step(self, batch, batch_idx):

        x, y = batch

        out_dict = self.model(x, y)

        # compute loss and get a dictionary containing all metrics (for each batch in each epoch)
        total_loss, metrics_dict = self.calculate_loss(
            x, y, out_dict,
            epoch=self.current_epoch,
            validation=False
        )

        # log all the training metrics
        for metric_name, metric_value in metrics_dict.items():
            self.log(f"train/{metric_name}", metric_value,
                    on_epoch=True, on_step=False, prog_bar = False)

        return total_loss

    def validation_step(self, batch, batch_idx):
      x, y = batch
      out_dict = self.model(x, y)

      # compute loss and get a dictionary containing all metrics (for each batch in each epoch)
      val_loss, loss_metrics = self.calculate_loss(
          x, y, out_dict,
          epoch=self.current_epoch,
          validation=True
      )

      # Calculate accuracy metrics (for each epoch)
      acc_metrics = self.calculate_val_metrics(x, y, out_dict)

      # Merge all metrics
      all_metrics = {**loss_metrics, **acc_metrics}

      # Log all validation metrics
      for metric_name, metric_value in all_metrics.items():
          self.log(f"val/{metric_name}", metric_value,
                  on_epoch=True, on_step=False, prog_bar = False)

      return val_loss

       # ===== ON VALIDATION EPOCH END =====
    def on_validation_epoch_end(self):
        """Log to console at end of each epoch"""
        epoch = self.current_epoch

        # Get logged metrics from trainer
        if hasattr(self.trainer, 'logged_metrics'):
            logger.info(f"\nEpoch {epoch}:")
            logger.info("Train metrics:")
            for key, value in self.trainer.logged_metrics.items():
                if key.startswith("train/"):
                    metric_name = key.replace("train/", "")
                    logger.info(f"  {metric_name:25s}: {value:.6f}")

            logger.info("Validation metrics:")
            for key, value in self.trainer.logged_metrics.items():
                if key.startswith("val/"):
                    metric_name = key.replace("val/", "")
                    logger.info(f"  {metric_name:25s}: {value:.6f}")

            # Track best accuracy
            if "val/y_accuracy" in self.trainer.logged_metrics:
                current_acc = self.trainer.logged_metrics["val/y_accuracy"]
                if current_acc > self.best_accuracy:
                    self.best_accuracy = current_acc
                    logger.info(f"  ⭐ New best accuracy: {self.best_accuracy:.6f}")



    def configure_optimizers(self):
        return torch.optim.Adam(self.model.parameters(), lr=self.cfg.lr)

    def calculate_loss(
        self,
        x: torch.Tensor,
        y: torch.Tensor,
        out_dict: Dict[str, torch.Tensor],
        epoch: int,
        validation: bool = False,
    ) -> Tuple[torch.Tensor, Dict[str, float]]:
        """
        Calculate all losses and return dict of metrics
        Returns: (total_loss, metrics_dict)
        """
        metrics = {}

       # ===== RECONSTRUCTION LOSSES =====
       # of both PRED and CL heads
        rec_loss_pred = (
            self.reconstruction_loss(
                out_dict["pred"]["x_reconstructed"],
                x * self.dev_std + self.dev_mean,
            )
            / 2
        )
        rec_loss_cl = (
            self.reconstruction_loss(
                out_dict["cl"]["x_reconstructed"],
                x * self.dev_std + self.dev_mean,
            )
            / 2
        )

        #save metrics in the dictionary
        metrics["reconstruction_loss_pred"] = rec_loss_pred.item()
        metrics["reconstruction_loss_cl"] = rec_loss_cl.item()

        reconstruction_loss = rec_loss_pred + rec_loss_cl

        # ===== PREDICTION LOSS (just pred head) =====
        y_reg_epoch = self.y_reg_values[epoch]
        if y_reg_epoch > 0:
            y_pred = out_dict["pred"]["y_pred"]
            loss_y = self.loss_y(y_pred, y.type_as(y_pred)) / 2
        else:
            loss_y = torch.tensor(0, dtype=torch.float, device=self.device)


        #save metric in the dictionary
        metrics["prediction_loss"] = loss_y.item()

        # ===== KL DIVERGENCES (Training only) =====
        # calculate kl div / loss on prior (KL for each z, both core and style for both pred and cl)
        beta_reg_epoch = self.beta_reg_values[epoch]
        total_z_loss = torch.tensor(0, dtype=torch.float, device=self.device)

        if not validation and beta_reg_epoch > 0:
            # Log each KL divergence individually
            #compute each KL, add it to the total z_loss, save each loss in the dictionary
            for name_z_loss in ["prior_z_core_y_cl", "prior_z_style_cl",
                               "prior_z_core_pred", "prior_z_style_pred"]:
                tmp_z_loss = latent_kl_divergence(
                    name_z_loss, out_dict,
                    self.model.z_core_dim,
                    self.model.z_style_dim
                )
                total_z_loss += tmp_z_loss
                metrics[name_z_loss] = tmp_z_loss.item()

        # ===== SPARSITY REGULARIZATION =====
        # calculate sparsity regularization on prediction and decoder weights relative to z_core
        if self.group_sparsity_reg > 0:
            pred_weights = self.model.get_prediction_weights()
            decoder_weights = self.model.get_decoder_first_linear_layer()
            decoder_weights = decoder_weights[
                :, : self.model.z_core_dim
            ]  # get only weights relative to z_core

            all_weights = torch.cat([pred_weights, decoder_weights], dim=0)
            group_sparsity = all_weights.norm(p="fro", dim=0).sum()
        else:
            group_sparsity = torch.tensor(0, dtype=torch.float, device=self.device)

        #save it into the dictionary
        metrics["group_sparsity"] = group_sparsity.item()

        # ===== TOTAL LOSS =====
        total_loss = (
            reconstruction_loss
            + beta_reg_epoch * total_z_loss
            + y_reg_epoch * loss_y
            + self.group_sparsity_reg * group_sparsity
        )
        metrics["total_loss"] = total_loss.item()

        return total_loss, metrics

    # function to compute accuracy (only for validation of classification head)
    def calculate_val_metrics(
        self, x: torch.Tensor, y: torch.Tensor, out_dict: Dict[str, torch.Tensor]
    ) -> Dict[str, float]:
        """
        Calculate accuracy metrics (validation only)
        Returns: metrics_dict
        """
        metrics = {}

        #get predictions from the prediction model
        y_pred = torch.sigmoid(out_dict["pred"]["y_pred"])

        #compute accuracy (for the single labels and the overall)
        single_label_acc, acc_mean = accuracy(y_pred, y)

        # Log per-label accuracy
        for i in range(self.model.n_outputs):
            metrics[f"y_accuracy{i}"] = single_label_acc[i].item()

        # Log mean accuracy
        metrics["y_accuracy"] = acc_mean.item()

        return metrics


In [ ]:
class CLAPDataModule(LightningDataModule):
  def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.setup_done = False

  def setup(self, stage= None):
    if not self.setup_done:
        self.train_dataset, self.test_dataset, \
        _, _, _ = get_datasets(self.cfg.dataset.dataset_name)
        self.setup_done = True

  def train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self.cfg.batch_size,
        drop_last=True,
        num_workers=5,
        shuffle=True)

  def test_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=self.cfg.batch_size,
        num_workers=5,
        shuffle=False)

  def val_dataloader(self):
    return DataLoader(
        self.test_dataset,
        batch_size=self.cfg.batch_size,
        num_workers=5,
        shuffle=False)


In [ ]:
#a = CLAPDataModule(cfg)

In [ ]:
#a.setup()

In [ ]:
#a.train_dataset

In [ ]:
#prova = LightningCLAP(
#        cfg,
 #       a.train_dataset,
  #      a.test_dataset
   #     )

In [ ]:
#llogger = L.pytorch.loggers.wandb.WandbLogger(project = cfg.wandb_project)
#trainer = L.Trainer(
 #     max_epochs=cfg.n_epochs_start + cfg.n_epochs_beta + cfg.n_epochs_end,
  #    accelerator="auto",
   #   logger = llogger
  #)

In [ ]:
#trainer.fit(prova, datamodule=a)

# Main

In [ ]:
if __name__ == "__main__":

    # cfg creation
    cfg = Config()

     # Setup
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger(__name__)
    logger.info(f"Starting training with config: {cfg}")

    #if cfg.seed is not None:
     #   torch.manual_seed(cfg.seed)

    # set seed
    if cfg.seed is not None:
      seed_everything(cfg.seed, workers=True)

    # load datasets, model, optimizer and trainer
    data_module = CLAPDataModule(cfg)
    data_module.setup()

    # set up the model
    model = LightningCLAP(
        cfg,
        data_module.train_dataset,
        data_module.test_dataset
    )

    # setup WandB logger
    llogger = L.pytorch.loggers.wandb.WandbLogger(project = cfg.wandb_project, config = cfg, log_model = "all")

    #Setup checkpoint callback (save the best and the latest value of classification accuracy and the loss)
    checkpoint_callback = [
        ModelCheckpoint(
            monitor="val/y_accuracy",
            mode="max", # the highest the better
            save_top_k=1, #best value
            save_last = True, #last value
            dirpath="checkpoints",
            filename="best-accuracy"
            ),
        ModelCheckpoint(
            monitor="val/total_loss",
            mode="min", #the lower the better
            save_top_k=1,
            save_last = True,
            dirpath="checkpoints",
            filename="best-loss"
            )
        ]
    #compute the total number of epochs
    total_epochs = cfg.n_epochs_start + cfg.n_epochs_beta + cfg.n_epochs_end

    trainer = L.Trainer(
        max_epochs=cfg.n_epochs_start + cfg.n_epochs_beta + cfg.n_epochs_end,
        accelerator="auto",
        logger = llogger,
        callbacks=checkpoint_callback
        )
    logger.info(f"Trainer initialized for {total_epochs} epochs")

    # Train
    logger.info("Starting training...")
    trainer.fit(model, datamodule=data_module)

    logger.info(f"Training completed! Best accuracy: {model.best_accuracy:.6f}")

INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _

INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name   ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model  │ CLAP              │  1.4 M │ train │     0 │
│ 1 │ loss_y │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.4 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.4 M                                                                                                
Total estimated model params size (MB): 5                                                                          
Modules in train mode: 66                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

...

wandb: WARNING Artifact "model-11ts0skt" already exists with the same content. No new version will be created.

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
  File "/usr/local/lib/python3.12/dist-packages/lightning/pytorch/loops/fit_loop.py", line 465, in advance
    self.epoch_loop.run(self._data_fetcher)
  File

TypeError: object of type 'NoneType' has no len()

In [ ]:
wandb.finish()

epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇██
train/group_sparsity,█▄▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂
train/prediction_loss,███▇▅▅▅▅▅▅▅▅▅▄▄▄▄▄▄▄▄▄▄▄▄▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁
train/prior_z_core_pred,█▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/prior_z_core_y_cl,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/prior_z_style_cl,█▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/prior_z_style_pred,█▇▆▅▅▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
train/reconstruction_loss_cl,█▇▇▄▄▃▂▄▄▅▃▂▂▂▄▃▄▄▃▂▂▄▂▃▁▄▃▂▄▂▃▂▂▂▃▃▂▃▄▂
train/reconstruction_loss_pred,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total_loss,█▃▃▂▁▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▂▂▁▂▁▂▂▂▂▂▂▂▃▂▂▂▂▂▃▂
+11,...


In [ ]:
/content/small_norb/smallnorb-5x01235x9x18x6x2x96x96-testing-cat.mat

SyntaxError: invalid decimal literal (2006946761.py, line 1)

In [ ]:
print(callbacks)
print(type(callbacks[0]))

found an error: **ValueError: Expected parameter scale (Tensor of shape (123, 10)) of distribution Normal(loc: torch.Size([123, 10]), scale: torch.Size([123, 10])) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values**


The error indicates that the standard deviation (scale) of the Normal distribution is becoming too close to zero, which is not allowed. This can be fixed by clamping the log_var values to prevent them from becoming too small, ensuring the scale is always positive.